# Stage 4 — Baseline ASR (Whisper large-v3)

**Attach:** `sarvam-diar-code`, `sarvam-diar-audio`, `sarvam-diar-stage3`.
**Settings:** GPU (T4) on, Internet on. No `HF_TOKEN` — the model is public.

This stage turns audio into **words with timestamps**, and nothing else. It never
sees a speaker label and never sees a diarization hypothesis. Attribution is a
separate CPU stage, so a single ASR run is reused across every diarization system
and every Stage 5 correction — which is what makes a cpWER delta attributable to
the labelling rather than to the ASR having been fed different audio.

### Why this is its own notebook

Whisper and IndicConformer want incompatible CUDA stacks. `onnxruntime-gpu`
installs its own `nvidia-cudnn-cu12`, which replaces the cuDNN that CTranslate2
(the runtime behind faster-whisper) was built against. The result is not an
error: Whisper silently falls back to CPU and a run that should take minutes
sits on an idle GPU for a quarter of an hour saying nothing.

Rather than fight that, each system gets its own session. They share nothing at
runtime — separate manifests, separate output directories, neither reads the
other — so the split costs nothing and removes a whole class of silent failure.
Save each notebook's output as a dataset; `stage4_attribute.py` is CPU-only and
attaches both.

### Why faster-whisper rather than WhisperX

WhisperX refines timestamps with per-language wav2vec2 alignment models, which do
not exist for most of the nine Indic scripts in this corpus. Whisper's own
cross-attention DTW word timestamps are coarser but exist for every language
here, and a metric that silently degrades for some languages and not others is
worse than one that is uniformly approximate.

Three decode settings are deliberate. `condition_on_previous_text=False` stops a
single hallucinated segment from seeding a loop across the rest of a long clip.
`vad_filter=False` because diarization owns the speech/non-speech decision — an
ASR-side VAD would delete words that the attribution stage is meant to score.
Segments that trip the compression-ratio or no-speech thresholds are **counted
and logged, never dropped**: a silently discarded segment is an invisible
deletion that inflates the miss rate with nothing recording why.

`temperature=0.0` disables Whisper's temperature fallback, and that one is a
correction rather than a precaution — the first corpus run left the default in
place. The fallback re-decodes a suspect segment at rising temperature, and above
zero it samples unseeded, so the transcript is a different draw each run. On
`Tlha36rSd5o` (318 reference words) three identical calls returned 87, 84 and 100
words; with the fallback off, 211 words, the same 211 every time. It was both
irreproducible and worse, because the loop keeps a sampled draw in preference to
the greedy decode it started from. The diagnostics at the end of this notebook
are what found it.

**Install only faster-whisper here.** Adding `onnxruntime-gpu` to this session replaces the cuDNN CTranslate2 needs and Whisper drops to CPU with no error.

In [1]:
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 77.6 MB/s eta 0:00:00


In [2]:
import pathlib, shutil

ROOT  = pathlib.Path("/kaggle/input")
CODE  = next(p.parent for p in ROOT.rglob("stage4_asr.py"))
AUDIO = next(p.parent for p in ROOT.rglob("*.wav"))
WORK  = pathlib.Path("/kaggle/working/data")
WORK.mkdir(parents=True, exist_ok=True)

for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")

# Restore anything already produced -- Stage 3 RTTMs, and the other ASR system's
# words if its dataset is attached. Nothing here is required by this notebook;
# it is what lets stage4_attribute.py run later without re-attaching everything.
for src in sorted(ROOT.rglob("data")):
    if src.is_dir() and any((src / d).exists() for d in ("hyp", "ref", "asr")):
        shutil.copytree(src, WORK, dirs_exist_ok=True)
        print("restored", src)

print("CODE   :", CODE)
print("AUDIO  :", AUDIO, len(list(AUDIO.glob("*.wav"))), "wavs")
print("scripts:", sorted(p.name for p in pathlib.Path("/kaggle/working").glob("*.py")))
for sub in ("hyp", "asr"):
    d = WORK / sub
    if d.exists():
        for x in sorted(d.glob("*")):
            n = len(list(x.rglob("*.rttm"))) + len(list(x.rglob("*.json")))
            print(f"  {sub}/{x.name}: {n}")

CODE   : /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code
AUDIO  : /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav 99 wavs
scripts: ['build_notebooks.py', 'stage1_extract.py', 'stage2_parse_refs.py', 'stage3_diarize.py', 'stage3_score.py', 'stage4_asr.py', 'stage4_attribute.py', 'stage4_score.py', 'stage5_correct.py']


## Smoke test — 2 clips

- `device=cuda` and `faster-whisper large-v3 (float16)`
- **RTF around 0.2–0.3.** Near 1.0, or a cell that sits for 15 minutes with an
  idle GPU, means CTranslate2 is on CPU
- native script in the words, not transliteration

In [3]:
import os, shutil, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [4]:
!python stage4_asr.py --system whisper --data data --wav-dir {AUDIO} --limit 2

[env ] device=cuda
[plan] 99 clips: 0 done, 99 pending, running 2 now
[env ] faster-whisper large-v3 (float16)
[  1/2] 0AEEA8NyVwY__000011000_000609000         ok     1039 words  rtf=0.1848
[  2/2] 0SoItGfM_sY__000007000_000088000         ok       74 words  rtf=0.2426

[done] ok=2 fail=0
[done] 0.19 h audio in 2.2 min (mean RTF 0.1916)


In [5]:
import json, glob

files = sorted(glob.glob("/kaggle/working/data/asr/whisper/words/*.json"))
print(len(files), "clips transcribed")
d = json.load(open(files[0], encoding="utf-8"))
w = d["words"]
print("clip     :", d["clip_id"][:44], f'{d["duration"]:.1f}s')
print("lang     :", d.get("lang"), d.get("lang_counts", ""))
print("words    :", len(w))
print("first    :", w[:6])
print("last     :", w[-3:])
print("text     :", " ".join(x["w"] for x in w[:40]))
print("span     :", w[0]["start"], "->", w[-1]["end"], "of", d["duration"], "s")
print("monotonic:", all(a["start"] <= b["start"] for a, b in zip(w, w[1:])))
print("in bounds:", w[-1]["end"] <= d["duration"] + 1)

2 clips transcribed
clip     : 0AEEA8NyVwY__000011000_000609000 598.0s
lang     : mr 
words    : 1039
first    : [{'w': 'नमस्कार', 'start': 0.0, 'end': 1.34}, {'w': 'मैं', 'start': 1.34, 'end': 1.74}, {'w': 'गौरो', 'start': 1.74, 'end': 1.92}, {'w': 'जोशी', 'start': 1.92, 'end': 2.2}, {'w': 'और', 'start': 2.2, 'end': 2.96}, {'w': 'मैं', 'start': 2.96, 'end': 3.26}]
last     : [{'w': 'दोन', 'start': 596.54, 'end': 596.72}, {'w': 'गुष्टी', 'start': 596.72, 'end': 596.92}, {'w': 'एक्जांपल्स', 'start': 596.92, 'end': 597.64}]
text     : नमस्कार मैं गौरो जोशी और मैं अमोल करहडकर और तुम्हारे सग्यांच कॉफी क्रिकेट और परच काई वाज़े CCBK वर स्वागत मित्रां नो आणि महित्र निन्नो आज अपन आजुन एक CCBK स्पेशल चा भाग गेवन तुमचा समोर येत आहेट आणि CCB
span     : 0.0 -> 597.64 of 598.0 s
monotonic: True
in bounds: True


## Full run

The first corpus run measured mean RTF 0.547 over 12.28 hours — **6.6 hours of
T4 time**, the longest single job in the project. Expect this pass to be quicker:
`temperature=0.0` removes up to five re-decodes per suspect segment, and those
re-decodes were most of the variance (per-clip RTF ranged 0.09 to 1.42). Start it
with plenty of session left regardless.

It is resumable, so a timeout costs only the clip in flight. **That also means it
will skip every clip already on disk** — set `WIPE = True` in the fresh-start
cell before rerunning, or the old stochastic transcripts survive and you score a
mixture of two decode settings.

In [6]:
WIPE = True   # set False once a clean run exists and you want to resume it

import pathlib, shutil

# The run is resumable, so a leftover clip is silently kept. After a decode
# change that is the dangerous case: the corpus ends up half one setting and
# half the other, and nothing in the output records the split.
src = pathlib.Path("/kaggle/working/stage4_asr.py").read_text(encoding="utf-8")
assert "temperature=0.0" in src, (
    "stale stage4_asr.py -- this copy still uses the default temperature "
    "fallback, which samples unseeded. Re-upload sarvam-diar-code, restart the "
    "session, and re-run the setup cell above"
)
print("stage4_asr.py: temperature=0.0 present")

out = pathlib.Path("/kaggle/working/data/asr/whisper")
if WIPE and out.exists():
    n = len(list((out / "words").glob("*.json")))
    shutil.rmtree(out)
    print(f"wiped {n} clips of previous whisper output")
print("present now:", sorted(p.name for p in
      pathlib.Path("/kaggle/working/data/asr").glob("*")))

stage4_asr.py: temperature=0.0 present
wiped 2 clips of previous whisper output
present now: []


In [7]:
!python stage4_asr.py --system whisper --data data --wav-dir {AUDIO}

[env ] device=cuda
[plan] 99 clips: 0 done, 99 pending, running 99 now
[env ] faster-whisper large-v3 (float16)
[  1/99] 0AEEA8NyVwY__000011000_000609000         ok     1039 words  rtf=0.193
[  2/99] 0SoItGfM_sY__000007000_000088000         ok       74 words  rtf=0.2404
[  3/99] 0VEwL9XZ0LY__000261000_000557000         ok      722 words  rtf=0.2685
[  4/99] 0esIFSOAcFs__000000000_000913000         ok      722 words  rtf=0.2156
[  5/99] 0p6cktLGIfY__000012000_000930000         ok      631 words  rtf=0.2113
[  6/99] 13VBh0Z6QmE__000000000_000904000         ok     1762 words  rtf=0.3216
[  7/99] 1LFl5JEipII__000000000_000597000         ok      563 words  rtf=0.216
[  8/99] 2HGP34TNvjg__000084000_000194000         ok      241 words  rtf=0.3321
[  9/99] 2LN6vb7EBn0__000098000_000394000         ok      420 words  rtf=0.2493
[ 10/99] 2T4pjueLrsk__000677000_001290000         ok     1662 words  rtf=0.0996
[ 11/99] 2gGHQgj9yno__000000000_000099000         ok       51 words  rtf=0.202
[ 12/99] 2i

In [8]:
import json, pathlib

mf = pathlib.Path("/kaggle/working/data/asr/whisper/manifest.jsonl")
recs = [json.loads(l) for l in mf.read_text(encoding="utf-8").splitlines() if l.strip()]
ok = [r for r in recs if r["status"] == "ok"]
print(f"whisper: {len(ok)} ok / {len(recs)} records, "
      f"{sum(r['n_words'] for r in ok):,} words")
for r in recs:
    if r["status"] != "ok":
        print("  fail:", r["clip_id"][:36], r.get("error", "")[:100])

rtfs = [r["rtf"] for r in ok if r.get("rtf")]
if rtfs:
    print(f"rtf: min {min(rtfs):.4f}  median {sorted(rtfs)[len(rtfs)//2]:.4f}  max {max(rtfs):.4f}")

langs = {}
for r in ok:
    langs[r.get("lang")] = langs.get(r.get("lang"), 0) + 1
print("languages:", dict(sorted(langs.items(), key=lambda kv: -kv[1])))

whisper: 99 ok / 99 records, 54,909 words
rtf: min 0.0716  median 0.2160  max 0.4146
languages: {'mr': 14, 'bn': 12, 'te': 12, 'hi': 11, 'gu': 10, 'ta': 10, 'kn': 9, 'pa': 7, 'ml': 7, 'en': 4, 'ne': 3}


## Save

**Save Version → Quick Save**, then Output tab → **New dataset**, named
`sarvam-diar-asr-whisper`.

In [9]:
!du -sh /kaggle/working/data/asr/* 2>/dev/null
!find /kaggle/working/data/asr -name "*.json" | wc -l

3.2M	/kaggle/working/data/asr/whisper
99


## Diagnostic - the word deficit

Whisper returned 54,577 words against 126,583 in the reference (ratio 0.43,
median 0.36 per clip). Most of that is genuine error, but the deficit is not
uniform: a tail of clips came back at 0.10-0.18 while spending near real time on
the GPU. Slow *and* near-empty points at windows being skipped inside the decode
loop rather than at fast wrong answers.

This cell is **diagnostic only** -- it writes nothing to `data/asr` and costs
about a minute of GPU. Rerun the corpus with the setting changed only if the
gain here is large.

In [10]:
# Does faster-whisper's internal no-speech skip explain the word deficit?
#
# vad_filter=False turns off the *external* Silero VAD, but CTranslate2 still
# applies no_speech_threshold=0.6 inside the decode loop: when a window looks
# like non-speech AND its avg_logprob is low, the window is skipped and emits
# nothing. That is invisible from outside -- the clip just comes back short.
#
# These four clips returned ~10-18% of the reference word count on the corpus
# run while spending near-real-time on the GPU, which is the signature of
# skipping rather than of fast, confident, wrong decoding.

import soundfile as sf
from faster_whisper import WhisperModel

PROBE = {
    'EmsKkNN2Me4__000035000_000095000':  180,
    'PRAzUz0GANs__000223000_000283000':  197,
    'Tlha36rSd5o__000240000_000374000':  318,
    'BGAAfht5dYw__000000000_000612000': 2149,
}

m = WhisperModel('large-v3', device='cuda', compute_type='float16')

def count(pcm, **kw):
    segs, _ = m.transcribe(pcm, word_timestamps=True, vad_filter=False,
                           condition_on_previous_text=False, **kw)
    return sum(len([w for w in (s.words or []) if w.word.strip()]) for s in segs)

hdr = ('clip', 'ref', 'default', 'ns=None', 'gain')
print('%-34s %6s %8s %8s %6s' % hdr)
for clip, ref in PROBE.items():
    pcm, sr = sf.read(str(AUDIO / (clip + '.wav')), dtype='float32')
    if pcm.ndim > 1:
        pcm = pcm.mean(1)
    a = count(pcm)
    b = count(pcm, no_speech_threshold=None)
    print('%-34s %6d %8d %8d %+6d' % (clip[:34], ref, a, b, b - a))

# Reading it: if ns=None recovers most of the gap, the deficit is a decode
# setting and the corpus run should be repeated with it off. If both columns
# stay near the default, Whisper genuinely cannot transcribe these clips and
# 86.68 WER is the honest number -- report it and move on.

clip                                  ref  default  ns=None   gain
EmsKkNN2Me4__000035000_000095000      180       55       91    +36
PRAzUz0GANs__000223000_000283000      197       20       20     +0
Tlha36rSd5o__000240000_000374000      318       96      116    +20
BGAAfht5dYw__000000000_000612000     2149      377      490   +113


## Diagnostic - is the decode reproducible?

The probe above returned word counts well above what the corpus run recorded for
the same clips, on identical audio and identical settings. That is not a
threshold question, so this cell asks the prior one: does Whisper return the same
transcript twice?

Roughly two minutes of GPU. If the default decode turns out to be stochastic,
every Whisper figure in the results table carries unmeasured variance and the
table needs a footnote at minimum.

In [11]:
# Is the decode even reproducible?
#
# The no-speech probe returned 47 and 117 words for two clips that the corpus
# run scored at 18 and 34 -- same model, same audio (read_wav's int16/32768 is
# bit-identical to soundfile float32), same three decode arguments. Nothing in
# the pipeline explains a 3.4x swing, which leaves the decoder itself.
#
# faster-whisper's default temperature is [0, 0.2, 0.4, 0.6, 0.8, 1.0]. A
# segment that trips the compression-ratio or avg-logprob threshold is
# re-decoded at the next temperature, and above zero that samples rather than
# taking the argmax -- unseeded. On clips where the fallback fires constantly
# the transcript is a different draw every run.
#
# temperature=0.0 disables the fallback outright: greedy, deterministic,
# repeatable. For a benchmark a reproducible number beats a slightly better
# one that no one else can reproduce.

import soundfile as sf
from faster_whisper import WhisperModel

CLIPS = ['Tlha36rSd5o__000240000_000374000',
         'EmsKkNN2Me4__000035000_000095000']
REPEATS = 3

m = WhisperModel('large-v3', device='cuda', compute_type='float16')

def count(pcm, **kw):
    segs, _ = m.transcribe(pcm, word_timestamps=True, vad_filter=False,
                           condition_on_previous_text=False, **kw)
    return sum(len([w for w in (s.words or []) if w.word.strip()]) for s in segs)

print('%-34s %-9s %s' % ('clip', 'setting', 'word counts over %d runs' % REPEATS))
for clip in CLIPS:
    pcm, _ = sf.read(str(AUDIO / (clip + '.wav')), dtype='float32')
    if pcm.ndim > 1:
        pcm = pcm.mean(1)
    for label, kw in (('default', {}), ('temp=0', {'temperature': 0.0})):
        runs = [count(pcm, **kw) for _ in range(REPEATS)]
        spread = max(runs) - min(runs)
        flag = 'STOCHASTIC' if spread else 'stable'
        print('%-34s %-9s %-22s spread %4d  %s'
              % (clip[:34], label, runs, spread, flag))

# Reading it: 'default' spread > 0 confirms the fallback is sampling, and every
# Whisper number in the results table inherits that variance. 'temp=0' must be
# stable across all three runs -- if it is not, the non-determinism is coming
# from somewhere else and changing temperature will not fix it.
#
# If temp=0 is stable AND its counts are not much worse than the default's
# best draw, set temperature=0.0 in stage4_asr.py and rerun the corpus. That is
# the only finding so far that justifies spending the 6.6 h again.

clip                               setting   word counts over 3 runs
Tlha36rSd5o__000240000_000374000   default   [75, 71, 63]           spread   12  STOCHASTIC
Tlha36rSd5o__000240000_000374000   temp=0    [211, 211, 211]        spread    0  stable
EmsKkNN2Me4__000035000_000095000   default   [19, 32, 132]          spread  113  STOCHASTIC
EmsKkNN2Me4__000035000_000095000   temp=0    [117, 117, 117]        spread    0  stable
